In [60]:
!pip install yfinance


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [61]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("FinSight-RAG-SP500") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.8


In [62]:
import datetime
today = datetime.date.today()

In [63]:
import os

p1 = f"./storage/raw/sp500_companies/extraction_date={today}/*.parquet"
p2 = "./storage/raw/sp500_companies"

if os.path.exists(p1):
    df = spark.read.format("parquet").load(p1)
elif os.path.exists(p2):
    df = spark.read.format("parquet").load(p2)
else:
    raise FileNotFoundError(f"Path not found: tried {p1} and {p2}")

In [64]:
df.show()

+------+--------------------+--------------------+--------------------+--------------------+----------+---------------+
|ticker|        company_name|              sector|        sub_industry|        headquarters|date_added|extraction_date|
+------+--------------------+--------------------+--------------------+--------------------+----------+---------------+
|   PGR|Progressive Corpo...|          Financials|Property & Casual...|Mayfield Village,...|1997-08-04|     2026-07-03|
|   PLD|            Prologis|         Real Estate|    Industrial REITs|San Francisco, Ca...|2003-07-17|     2026-07-03|
|   PRU|Prudential Financial|          Financials|Life & Health Ins...|  Newark, New Jersey|2002-07-22|     2026-07-03|
|   PEG|Public Service En...|           Utilities|  Electric Utilities|  Newark, New Jersey|1957-03-04|     2026-07-03|
|   PTC|            PTC Inc.|Information Techn...|Application Software|Boston, Massachus...|2021-04-20|     2026-07-03|
|   PSA|      Public Storage|         Re

In [65]:
from pyspark.sql.functions import collect_list

In [66]:
df = df.groupBy("sector").agg(collect_list("ticker").alias("companies"))

In [67]:
df.show()

+--------------------+--------------------+
|              sector|           companies|
+--------------------+--------------------+
|         Health Care|[DGX, REGN, RMD, ...|
|Communication Ser...|[TMUS, TTWO, TKO,...|
|              Energy|[SLB, TRGP, TPL, ...|
|Information Techn...|[PTC, QCOM, Q, RO...|
|         Real Estate|[PLD, PSA, O, REG...|
|           Materials|[SHW, SW, STLD, V...|
|Consumer Discreti...|[PHM, RL, ROST, R...|
|           Utilities|[PEG, SRE, SO, VS...|
|    Consumer Staples|[SJM, SYY, TGT, T...|
|         Industrials|[PWR, RTX, RSG, R...|
|          Financials|[PGR, PRU, RJF, R...|
+--------------------+--------------------+



In [68]:
import os

In [69]:
# TEMP: TO SAVE THE DATA IN JSON FORMAT

os.makedirs(f"./storage/foundation/{today}", exist_ok=True)
df.write.mode("overwrite").json(f"./storage/foundation/{today}/companies")

In [70]:
import yfinance as yf

sector_stock_data = {}
sector_rows = df.select("sector", "companies").collect()

for row in sector_rows:
    sector = row["sector"]
    companies = row["companies"][:5]
    sector_stock_data[sector] = {}
    print(f"Sector: {sector}")
    if not companies:
        print("  No companies found for this sector.")
        continue

    for company in companies:
        print(f"  Company/Ticker: {company}")
        ticker = yf.Ticker(company)
        info = ticker.info or {}

        if not info:
            print("    No yfinance info found for this symbol.")
            continue

        details = {
            "shortName": info.get("shortName"),
            "longName": info.get("longName"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "marketCap": info.get("marketCap"),
            "regularMarketPrice": info.get("regularMarketPrice"),
            "previousClose": info.get("previousClose"),
            "website": info.get("website"),
            "summary": info.get("longBusinessSummary", "")[:200]
        }

        sector_stock_data[sector][company] = details

        for key, value in details.items():
            print(f"    {key}: {value}")
    print()

Sector: Health Care
  Company/Ticker: DGX
    shortName: Quest Diagnostics Incorporated
    longName: Quest Diagnostics Incorporated
    sector: Healthcare
    industry: Diagnostics & Research
    marketCap: 23966330880
    regularMarketPrice: 216.505
    previousClose: 216.02
    website: https://www.questdiagnostics.com
    summary: Quest Diagnostics Incorporated provides diagnostic testing and services in the United States. The company develops and delivers diagnostic information services, such as routine, non-routine and advanc
  Company/Ticker: REGN
    shortName: Regeneron Pharmaceuticals, Inc.
    longName: Regeneron Pharmaceuticals, Inc.
    sector: Healthcare
    industry: Biotechnology
    marketCap: 67814600704
    regularMarketPrice: 646.845
    previousClose: 624.72
    website: https://www.regeneron.com
    summary: Regeneron Pharmaceuticals, Inc. discovers, invents, develops, manufactures, and commercializes medicines to treat various diseases worldwide. The company deve

In [71]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# Save the collected sector/company details to the requested foundation path
rows = []
for sector, companies in sector_stock_data.items():
    for company, details in companies.items():
        row = {"sector": sector, "company": company}
        row.update(details)
        rows.append(row)

if rows:
    details_schema = StructType([
        StructField("sector", StringType(), True),
        StructField("company", StringType(), True),
        StructField("shortName", StringType(), True),
        StructField("longName", StringType(), True),
        StructField("industry", StringType(), True),
        StructField("marketCap", LongType(), True),
        StructField("regularMarketPrice", DoubleType(), True),
        StructField("previousClose", DoubleType(), True),
        StructField("website", StringType(), True),
        StructField("summary", StringType(), True),
    ])

    details_df = spark.createDataFrame(rows, schema=details_schema)
    details_df.write.mode("overwrite").json(f"./storage/foundation/{today}/companies_data")
    print(f"Saved details to ./storage/foundation/{today}/companies_data")
else:
    print("No details were available to save.")

Saved details to ./storage/foundation/2026-07-03/companies_data


In [ ]:
# SECTORS = {
#     "Technology": ["AAPL", "MSFT"],
#     "Finance":    ["JPM", "BAC"],
#     "Energy":     ["XOM", "CVX"],
#     "Healthcare": ["JNJ", "UNH"],
#     "EV & Auto":  ["TSLA", "TM"]
# }

In [73]:
import yfinance as yf

for sector, tickers in SECTORS.items():
    print(f"Fetching data for {sector} sector...")
    for ticker in tickers:
        for sector, tickers in SECTORS.items():
            print(f"Fetching data for {sector} sector...")
            for ticker in tickers:
                stock_data = yf.download(ticker, period="1y")
                print(f"Data for {ticker}:\n{stock_data.head()}\n")
        print(f"Data for {ticker}:\n{stock_data.head()}\n")

Fetching data for Technology sector...
Fetching data for Technology sector...


[*********************100%***********************]  1 of 1 completed


Data for AAPL:
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2025-07-02  211.600540  212.496978  207.317529  208.084490  67941800
2025-07-03  212.706161  213.801806  210.973032  211.311684  34955800
2025-07-07  209.120361  215.375544  207.974912  211.839569  50229000
2025-07-08  209.180130  210.594517  207.626297  209.269786  42848900
2025-07-09  210.305679  210.494931  206.401171  208.702041  48749400



[*********************100%***********************]  1 of 1 completed


Data for MSFT:
Price            Close        High         Low        Open    Volume
Ticker            MSFT        MSFT        MSFT        MSFT      MSFT
Date                                                                
2025-07-02  487.198486  489.589393  484.827441  486.107197  16319600
2025-07-03  494.887054  496.166841  489.529852  489.896915  13984800
2025-07-07  493.775909  494.797746  491.305651  493.438607  13981600
2025-07-08  492.684631  494.252128  490.194512  493.299713  11846600
2025-07-09  499.520050  502.764127  495.779905  496.335465  18659500

Fetching data for Finance sector...


...

Data for UNH:
Price            Close        High         Low        Open    Volume
Ticker             UNH         UNH         UNH         UNH       UNH
Date                                                                
2025-07-02  299.537842  314.010261  298.437311  312.374058  18177200
2025-07-03  300.501984  304.105487  300.190353  301.758344   6161100
2025-07-07  295.788239  300.560425  293.343728  299.966350   9031200
2025-07-08  299.674164  300.258490  295.875865  296.294672   9613800
2025-07-09  295.009094  295.681099  290.957597  295.281790  12816700

Fetching data for EV & Auto sector...


[*********************100%***********************]  1 of 1 completed


Data for TSLA:
Price            Close        High         Low        Open     Volume
Ticker            TSLA        TSLA        TSLA        TSLA       TSLA
Date                                                                 
2025-07-02  315.649994  316.829987  303.820007  312.630005  119483700
2025-07-03  315.350006  318.450012  312.760010  317.989990   58042300
2025-07-07  293.940002  296.149994  288.769989  291.369995  131177900
2025-07-08  297.809998  304.049988  294.350006  297.000000  103246700
2025-07-09  295.880005  300.149994  293.549988  297.549988   75586800



[*********************100%***********************]  1 of 1 completed

Data for TM:
Price            Close        High         Low        Open  Volume
Ticker              TM          TM          TM          TM      TM
Date                                                              
2025-07-02  172.009995  172.289993  171.110001  171.440002  441900
2025-07-03  174.889999  175.250000  174.009995  174.389999  335000
2025-07-07  167.860001  170.699997  167.179993  170.199997  991800
2025-07-08  170.199997  170.320007  168.600006  168.600006  485700
2025-07-09  170.539993  170.619995  169.350006  170.190002  336000

Data for TM:
Price            Close        High         Low        Open  Volume
Ticker              TM          TM          TM          TM      TM
Date                                                              
2025-07-02  172.009995  172.289993  171.110001  171.440002  441900
2025-07-03  174.889999  175.250000  174.009995  174.389999  335000
2025-07-07  167.860001  170.699997  167.179993  170.199997  991800
2025-07-08  170.199997  170.320007 

In [ ]:
# # %pip install psycopg2-binary

# import psycopg2
# from psycopg2 import sql
# from getpass import getpass

# # Connect to PostgreSQL database
# _pg_host = "localhost"
# _pg_db = "stock_data"
# _pg_user = "root"

# # Prompt for password instead of hardcoding; press Enter to try connecting without a password (peer/trust auth)
# _pg_pass = getpass(f"Enter password for PostgreSQL user '{_pg_user}' (leave blank to try without password): ")

# try:
#     if _pg_pass:
#         conn = psycopg2.connect(host=_pg_host, database=_pg_db, user=_pg_user, password=_pg_pass)
#     else:
#         conn = psycopg2.connect(host=_pg_host, database=_pg_db, user=_pg_user)
# except psycopg2.OperationalError:
#     # Fallback: try the default 'postgres' user if 'root' fails
#     _pg_user = "postgres"
#     _pg_pass = getpass(f"Connection as 'root' failed. Enter password for '{_pg_user}' (leave blank to try without password): ")
#     if _pg_pass:
#         conn = psycopg2.connect(host=_pg_host, database=_pg_db, user=_pg_user, password=_pg_pass)
#     else:
#         conn = psycopg2.connect(host=_pg_host, database=_pg_db, user=_pg_user)
# cursor = conn.cursor()

# # Create table if it doesn't exist
# cursor.execute("""
#     CREATE TABLE IF NOT EXISTS stock_prices (
#         id SERIAL PRIMARY KEY,
#         ticker VARCHAR(10),
#         date TIMESTAMP,
#         open FLOAT,
#         high FLOAT,
#         low FLOAT,
#         close FLOAT,
#         volume BIGINT,
#         sector VARCHAR(50)
#     )
# """)

# # Insert stock_data into PostgreSQL
# for idx, row in stock_data.iterrows():
#     cursor.execute("""
#         INSERT INTO stock_prices (ticker, date, open, high, low, close, volume, sector)
#         VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
#     """, (ticker, idx, row['Open'], row['High'], row['Low'], row['Close'], row['Volume'], sector))

# conn.commit()
# cursor.close()
# conn.close()

# print(f"Data for {ticker} in {sector} sector stored successfully!")

OperationalError: connection to server at "localhost" (::1), port 5432 failed: fe_sendauth: no password supplied
